# STEP 5: LangGraph의 State, Node, Edge

이 노트북의 목표는 기업 분석 workflow에서 LangGraph가 **데이터를 어떤 순서로 전달하는지** 이해하는 것입니다.

`초기 State → 영업이익률 계산 Node → 요약 Node → 최종 State`

이번 단계에서는 가짜 재무 데이터를 사용합니다. OpenDART API, LLM, Tool, 조건 분기, Agent는 넣지 않습니다.

## 0. 왜 가짜 데이터를 쓰는가?

앞 단계에서는 OpenDART와 Tool Calling을 확인했습니다. 여기서는 외부 API나 LLM 결과가 아니라, **State가 Node를 거치며 어떻게 갱신되는가**만 분리해 관찰합니다.

이것은 최종 Agent pipeline이 아닙니다. 이후 실제 OpenDART 조회 결과를 State에 넣는 workflow로 확장하기 전의 최소 개념 예제입니다.

## 1. LangGraph 구성요소 불러오기

- `StateGraph`: State와 Node, Edge를 연결하는 그래프 설계 도구
- `START`, `END`: workflow의 시작과 종료를 표시하는 특수 지점
- `TypedDict`: State에 어떤 데이터가 들어가는지 읽기 쉽게 적는 타입 표기

In [1]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

## 2. State 정의

State는 workflow가 Node 사이에서 들고 다니는 공유 데이터입니다. `total=False`는 모든 항목이 시작부터 반드시 있어야 하는 것은 아니라는 타입 표기입니다.

예를 들어 처음에는 `operating_margin`과 `summary`가 없고, 각 Node가 계산한 값을 State에 추가합니다.

In [2]:
class FinancialWorkflowState(TypedDict, total=False):
    company_name: str
    revenue: int
    operating_income: int
    operating_margin: float
    summary: str

## 3. workflow의 초기 State

실제 프로젝트에서는 이 값이 OpenDART 조회 결과가 될 수 있습니다. 지금은 계산 흐름만 보기 위해 직접 넣습니다. 금액의 단위는 같다고 가정합니다.

In [3]:
initial_state: FinancialWorkflowState = {
    "company_name": "가상기업",
    "revenue": 1_000,
    "operating_income": 120,
}

initial_state

{'company_name': '가상기업', 'revenue': 1000, 'operating_income': 120}

## 4. 첫 번째 Node: 영업이익률 계산

Node는 State를 입력으로 받아 필요한 값을 계산하고, **변경하거나 추가할 값만 딕셔너리로 반환**합니다. State 전체를 직접 수정하지 않는다는 점을 확인하세요.

LangGraph가 반환된 `operating_margin`을 기존 State와 합칩니다.

In [4]:
def calculate_operating_margin(state: FinancialWorkflowState) -> dict[str, float]:
    margin = state["operating_income"] / state["revenue"] * 100
    return {"operating_margin": round(margin, 2)}

## 5. 두 번째 Node: 계산 결과를 요약

이 Node는 첫 번째 Node가 추가한 `operating_margin`을 읽습니다. 따라서 두 Node의 실행 순서는 Edge로 보장해야 합니다.

In [5]:
def create_summary(state: FinancialWorkflowState) -> dict[str, str]:
    summary = (
        f'{state["company_name"]}의 영업이익률은 '
        f'{state["operating_margin"]}%입니다.'
    )
    return {"summary": summary}

## 6. Graph에 Node와 Edge 연결

아래 코드는 다음 순서를 선언합니다.

`START → calculate_margin → create_summary → END`

`compile()`은 설계한 graph를 실제로 실행 가능한 객체로 만듭니다. 아직 실행은 하지 않습니다.

In [6]:
workflow = StateGraph(FinancialWorkflowState)

workflow.add_node("calculate_margin", calculate_operating_margin)
workflow.add_node("create_summary", create_summary)

workflow.add_edge(START, "calculate_margin")
workflow.add_edge("calculate_margin", "create_summary")
workflow.add_edge("create_summary", END)

app = workflow.compile()

## 7. Graph 구조를 텍스트로 확인

Mermaid 텍스트는 Graph의 연결을 표현하는 표기입니다. 지금은 그림을 만드는 기능을 추가하지 않고, Node와 Edge 이름이 선언한 순서와 맞는지만 읽어 봅니다.

In [7]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	calculate_margin(calculate_margin)
	create_summary(create_summary)
	__end__([<p>__end__</p>]):::last
	__start__ --> calculate_margin;
	calculate_margin --> create_summary;
	create_summary --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 8. Graph 실행

`invoke()`에 초기 State를 전달하면 LangGraph가 정해진 Edge 순서대로 Node를 실행하고, 각 Node의 반환값을 State에 합칩니다.

결과에서 처음 없던 `operating_margin`, `summary`가 추가됐는지 확인하세요.

In [8]:
final_state = app.invoke(initial_state)
final_state

{'company_name': '가상기업',
 'revenue': 1000,
 'operating_income': 120,
 'operating_margin': 12.0,
 'summary': '가상기업의 영업이익률은 12.0%입니다.'}

## 9. 결과를 직접 확인

이 확인은 정량 평가가 아니라, 이 최소 graph가 의도대로 State를 갱신했는지 보는 학습용 점검입니다.

In [9]:
assert final_state["operating_margin"] == 12.0
assert final_state["summary"] == "가상기업의 영업이익률은 12.0%입니다."

print("State가 두 Node를 거쳐 예상대로 갱신되었습니다.")

State가 두 Node를 거쳐 예상대로 갱신되었습니다.


## 이번 단계에서 기억할 점

- **State**: Node 사이에서 전달되는 공유 데이터이며, 기본적으로 데이터베이스나 장기 메모리가 아닙니다.
- **Node**: State를 읽고 필요한 변경값만 반환하는 작업 단위입니다.
- **Edge**: Node 실행 순서를 정합니다.
- **compile / invoke**: graph 설계를 실행 가능한 형태로 만들고, 실제 State를 넣어 실행합니다.

현재 graph는 직선 흐름입니다. 조건에 따라 다른 경로를 선택하는 Router와 반복 실행은 다음 학습 단계에서만 다룹니다.